In [1]:
import smote_variants as sv
# import sklearn.datasets as datasets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.kernel_approximation import RBFSampler
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import train_test_split
from sklearn import svm
from sklearn.metrics import classification_report
from sklearn import metrics
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (precision_score, recall_score,f1_score, accuracy_score,mean_squared_error,mean_absolute_error)
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import Normalizer
from pandas_ml import ConfusionMatrix

import argparse
from preprocess import preprocess
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score,precision_recall_curve, precision_score, recall_score, average_precision_score, roc_curve, auc, confusion_matrix, mean_squared_error,classification_report
import time

from keras.utils import to_categorical


Using TensorFlow backend.


In [2]:
trainset = pd.read_csv('./data/NSL-KDD/KDDTrain+.txt', sep=",", header=None)
testset = pd.read_csv('./data/NSL-KDD/KDDTest+.txt', sep=",", header=None)

processor = preprocess()
print("数据预处理....")
df_train, df_test, train_Normal, train_R2L, train_U2R, train_Dos, train_Probe,test_Normal, test_R2L, test_U2R, test_Dos, test_Probe,train_Attack,test_Attack = processor.create_df(df_train=trainset, df_test=testset)
# normal_df, R2L_df, R2L_df_train, R2L_df_test = processor.create_df(df_train=trainset, df_test=testset)
print("已完成数据预处理")

数据预处理....
已完成数据预处理


In [3]:
dt={'Dos':pd.Series([len(train_Dos),len(test_Dos)],index=['Train','Test']),
   'Probe':pd.Series([len(train_Probe),len(test_Probe)],index=['Train','Test']),
   'R2L':pd.Series([len(train_R2L),len(test_R2L)],index=['Train','Test']),
   'U2R':pd.Series([len(train_U2R),len(test_U2R)],index=['Train','Test']),
   'Normal':pd.Series([len(train_Normal),len(test_Normal)],index=['Train','Test']),
   'Total_attack':pd.Series([len(train_Attack),len(test_Attack)],index=['Train','Test']),
   'Total':pd.Series([len(df_train),len(df_test)],index=['Train','Test'])}
type_df=pd.DataFrame(dt)
cols = ['Dos','Probe','R2L','U2R','Normal','Total_attack','Total']
type_df = type_df[cols]
display(type_df)

,Dos,Probe,R2L,U2R,Normal,Total_attack,Total
Train,11656,45927,995,52,67343,58630,125973
Test,2421,7460,2885,67,9711,12833,22544


  Dos: 2   
  Probe:3  
  R2L:4   
  U2R : 1

In [4]:
#随机划分的全局二分类
from sklearn.model_selection import train_test_split

train_Bi = df_train
test_Bi = df_test

combined_data = pd.concat([train_Bi, test_Bi])
data_x = combined_data.drop(['attack_type'], axis=1) # droped label
data_y = combined_data.loc[:,['attack_type']]
# del combined_data # free mem
X_train, X_test, y_train, y_test = train_test_split(data_x, data_y, test_size=.20, random_state=42) # TODO




In [5]:
# X_train_Mult,y_train_Mult = processor.split_df(df_train)


selectedSmoteData = combined_data.query('attack_type == [1,2,4] ')
select_X = selectedSmoteData.drop(['attack_type'], axis=1)
select_y = selectedSmoteData.loc[:,['attack_type']]

# select_X = combined_data.drop(['attack_type'], axis=1) # droped label
# select_y = combined_data.loc[:,['attack_type']]
np_select_X = select_X.values
np_select_y = select_y.values.flatten()

oversampler= sv.MulticlassOversampling(sv.distance_SMOTE())
X_samp, y_samp= oversampler.sample(np_select_X, np_select_y)

2021-08-29 13:47:37,621:INFO:MulticlassOversampling: Running multiclass oversampling with strategy eq_1_vs_many_successive
2021-08-29 13:47:37,632:INFO:MulticlassOversampling: Sampling minority class with label: 1
2021-08-29 13:47:37,636:INFO:distance_SMOTE: Running sampling via ('distance_SMOTE', "{'proportion': 1.0, 'n_neighbors': 5, 'n_jobs': 1, 'random_state': None}")
2021-08-29 13:47:38,189:INFO:MulticlassOversampling: Sampling minority class with label: 4
2021-08-29 13:47:38,205:INFO:distance_SMOTE: Running sampling via ('distance_SMOTE', "{'proportion': 0.49787765293383268, 'n_neighbors': 5, 'n_jobs': 1, 'random_state': None}")


In [6]:
for i in np.unique(y_samp):
    print("class %d - samples: %d" % (i, np.sum(y_samp == i)))


class 1 - samples: 14077
class 2 - samples: 14077
class 4 - samples: 14077


In [7]:
remainData = combined_data.query('attack_type == [0,3] ')

remain_X = remainData.drop(['attack_type'], axis=1)
remain_y = remainData.loc[:,['attack_type']]



In [8]:
X_real_train = np.append(remain_X.values, X_samp, axis = 0)
y_real_train = np.where(np.append(remain_y.values, y_samp) == 0 ,0,1)
X_real_test = X_test.values
y_real_test = np.where(y_test.values == 0,0,1).flatten()

In [9]:
X_real_train

array([[ 0.        ,  0.5       ,  0.28985507, ...,  0.        ,
         0.05      ,  0.        ],
       [ 0.        ,  1.        ,  0.63768116, ...,  0.        ,
         0.        ,  0.        ],
       [ 0.        ,  0.5       ,  0.71014493, ...,  1.        ,
         0.        ,  0.        ],
       ..., 
       [ 0.00183055,  0.5       ,  0.64269722, ...,  0.        ,
         0.03245319,  0.03245319],
       [ 0.00359818,  0.5       ,  0.64418761, ...,  0.00568941,
         0.        ,  0.        ],
       [ 0.00109527,  0.5       ,  0.86956522, ...,  0.        ,
         0.00340288,  0.        ]])

In [10]:
print(X_real_train.shape)
print(y_real_train.shape)
print(X_real_test.shape) # test is larger... good 
print(y_real_test.shape)
# y_train.min()

(172672, 41)
(172672,)
(29704, 41)
(29704,)


In [11]:
X = X_real_train
Y = y_real_train
T = X_real_test
C = y_real_test

# X = X_train
# T = X_test
# Y = y_train
# C = y_test

In [12]:

# traindata = pd.read_csv('UNSW_NB15_training-set.csv', header=None)
# testdata = pd.read_csv('UNSW_NB15_testing-set.csv', header=None)
# traindata = pd.read_csv('kddtrain.csv', header=None)
# testdata = pd.read_csv('kddtest.csv', header=None)


# X = traindata.iloc[:,1:42]
# Y = traindata.iloc[:,0]
# C = testdata.iloc[:,0]
# T = testdata.iloc[:,1:42]


scaler = Normalizer().fit(X)
trainX = scaler.transform(X)

scaler = Normalizer().fit(T)
testT = scaler.transform(T)


traindata = np.array(trainX)
trainlabel = Y
# trainlabel = np.array(Y)

testdata = np.array(testT)
testlabel = np.array(C)
testlabel = testlabel.flatten()

model = LogisticRegression()
model.fit(traindata, trainlabel)


# make predictions
expected = testlabel
predicted = model.predict(testdata)

#predicted = predicted.reshape(len(predicted),1)
print(traindata.shape)
print(trainlabel.shape)
print(expected.shape)
print(predicted.shape)

print("***************************************************************")



D:\Anaconda3\envs\TF_36a\lib\site-packages\sklearn\linear_model\logistic.py:432: FutureWarning: Default solver will be changed to 'lbfgs' in 0.22. Specify a solver to silence this warning.
  FutureWarning)


(172672, 41)
(172672,)
(29704,)
(29704,)
***************************************************************


In [13]:
model = LogisticRegression()
model.fit(traindata, trainlabel)
print(model)

# make predictions
expected = testlabel
predicted = model.predict(testdata)

#predicted = predicted.reshape(len(predicted),1)
print(expected.shape)
print(predicted.shape)


cm = ConfusionMatrix(expected, predicted)

expected = np.array(expected)
predicted = np.array(predicted)
cm.print_stats()

np.savetxt('expected.txt', expected, fmt='%01d')
np.savetxt('predicted.txt',predicted , fmt='%01d')

print(cm)
print(expected.shape)
print(predicted.shape)
cm.stats()
print("***************************************************************")



D:\Anaconda3\envs\TF_36a\lib\site-packages\sklearn\linear_model\logistic.py:432: FutureWarning: Default solver will be changed to 'lbfgs' in 0.22. Specify a solver to silence this warning.
  FutureWarning)


LogisticRegression(C=1.0, class_weight=None, dual=False, fit_intercept=True,
          intercept_scaling=1, max_iter=100, multi_class='warn',
          n_jobs=None, penalty='l2', random_state=None, solver='warn',
          tol=0.0001, verbose=0, warm_start=False)
(29704,)
(29704,)
population: 29704
P: 14254
N: 15450
PositiveTest: 15114
NegativeTest: 14590
TP: 13351
TN: 13687
FP: 1763
FN: 903
TPR: 0.936649361583
TNR: 0.885889967638
PPV: 0.88335318248
NPV: 0.938108293352
FPR: 0.114110032362
FDR: 0.11664681752
FNR: 0.0633506384173
ACC: 0.910247778077
F1_score: 0.90922092073
MCC: 0.822000225858
informedness: 0.82253932922
markedness: 0.821461475831
prevalence: 0.479868031242
LRP: 8.20829985051
LRN: 0.0715107301488
DOR: 114.784170619
FOR: 0.0618917066484
Predicted  False   True  __all__
Actual                          
False      13687   1763    15450
True         903  13351    14254
__all__    14590  15114    29704
(29704,)
(29704,)
*********************************************************

In [14]:
# fit a Naive Bayes model to the data
model = GaussianNB()
model.fit(traindata, trainlabel)
print(model)
# make predictions
expected = testlabel
predicted = model.predict(testdata)

# expected = expected.flatten()
#predicted = predicted.reshape(len(predicted),1)
print(expected.shape)
print(predicted.shape)

print(type(expected))

cm = ConfusionMatrix(expected, predicted)

expected = np.array(expected)
predicted = np.array(predicted)



cm.print_stats()

np.savetxt('expected.txt', expected, fmt='%01d')
np.savetxt('predicted.txt',predicted , fmt='%01d')

print(cm)
print(expected.shape)
print(predicted.shape)
cm.stats()
print("***************************************************************")





# fit a k-nearest neighbor model to the data
model = KNeighborsClassifier()
model.fit(traindata, trainlabel)
print(model)
# make predictions
expected = testlabel
predicted = model.predict(testdata)
# summarize the fit of the model

#expected = expected.flatten()
print(expected.shape)
print(predicted.shape)

cm = ConfusionMatrix(expected, predicted)
expected = np.array(expected)
predicted = np.array(predicted)
cm.print_stats()
np.savetxt('expected.txt', expected, fmt='%01d')
np.savetxt('predicted.txt',predicted , fmt='%01d')
print(cm)
print(expected.shape)
print(predicted.shape)
cm.stats()


# cm = metrics.confusion_matrix(expected, predicted)
# print(cm)
# tpr = float(cm[0][0])/np.sum(cm[0])
# fpr = float(cm[1][1])/np.sum(cm[1])
# print("%.3f" %tpr)
# print("%.3f" %fpr)
# print("Accuracy")
# print("%.3f" %ACC)
# print("precision")
# print("%.3f" %precision)
# print("recall")
# print("%.3f" %recall)
# print("f-score")
# print("%.3f" %f1)
# print("fpr")
# print("%.3f" %fpr)
# print("tpr")
# print("%.3f" %tpr)
print("***************************************************************")



model = DecisionTreeClassifier()
model.fit(traindata, trainlabel)
print(model)
# make predictions
expected = testlabel
predicted = model.predict(testdata)
# summarize the fit of the model

#expected = expected.flatten()

cm = ConfusionMatrix(expected, predicted)
expected = np.array(expected)
predicted = np.array(predicted)
cm.print_stats()
np.savetxt('expected.txt', expected, fmt='%01d')
np.savetxt('predicted.txt',predicted , fmt='%01d')
print(cm)
print(expected.shape)
print(predicted.shape)
cm.stats()
print("***************************************************************")

print("AdaBoostClassifier")
model = AdaBoostClassifier(n_estimators=100)
model.fit(traindata, trainlabel)

# make predictions
expected = testlabel
predicted = model.predict(testdata)
# summarize the fit of the model

#expected = expected.flatten()

cm = ConfusionMatrix(expected, predicted)
expected = np.array(expected)
predicted = np.array(predicted)
cm.print_stats()
np.savetxt('expected.txt', expected, fmt='%01d')
np.savetxt('predicted.txt',predicted , fmt='%01d')
print(cm)
print(expected.shape)
print(predicted.shape)
cm.stats()

print("***************************************************************")



print("RandomForestClassifier")
model = RandomForestClassifier(n_estimators=100)
model = model.fit(traindata, trainlabel)

# make predictions
expected = testlabel
predicted = model.predict(testdata)
# summarize the fit of the model
#expected = expected.flatten()

cm = ConfusionMatrix(expected, predicted)
expected = np.array(expected)
predicted = np.array(predicted)
cm.print_stats()
np.savetxt('expected.txt', expected, fmt='%01d')
np.savetxt('predicted.txt',predicted , fmt='%01d')
print(cm)
print(expected.shape)
print(predicted.shape)
cm.stats()

print("end****end***********************************************************")




GaussianNB(priors=None, var_smoothing=1e-09)
(29704,)
(29704,)
<class 'numpy.ndarray'>
population: 29704
P: 14254
N: 15450
PositiveTest: 13197
NegativeTest: 16507
TP: 11876
TN: 14129
FP: 1321
FN: 2378
TPR: 0.833169636593
TNR: 0.914498381877
PPV: 0.899901492764
NPV: 0.855939904283
FPR: 0.085501618123
FDR: 0.100098507236
FNR: 0.166830363407
ACC: 0.875471316994
F1_score: 0.865250810535
MCC: 0.751743599645
informedness: 0.74766801847
markedness: 0.755841397047
prevalence: 0.479868031242
LRP: 9.74448969369
LRN: 0.182428276214
DOR: 53.4154567258
FOR: 0.144060095717
Predicted  False   True  __all__
Actual                          
False      14129   1321    15450
True        2378  11876    14254
__all__    16507  13197    29704
(29704,)
(29704,)
***************************************************************
KNeighborsClassifier(algorithm='auto', leaf_size=30, metric='minkowski',
           metric_params=None, n_jobs=None, n_neighbors=5, p=2,
           weights='uniform')
(29704,)
(29704,)
po

In [15]:
#model = svm.OneClassSVM( kernel='linear', random_state=0)
# model = svm.LinearSVC(C=20)
model = svm.SVC(kernel='linear')#调参
#model = svm.SVC(kernel='linear', C=1.0, random_state=0)
model = model.fit(traindata, trainlabel)

# make predictions
expected = testlabel
predicted = model.predict(testdata)
# summarize the fit of the model
expected = expected.flatten()
print(expected.shape)
print(predicted.shape)

cm = ConfusionMatrix(expected, predicted)
expected = np.array(expected)
predicted = np.array(predicted)
cm.print_stats()
np.savetxt('expected.txt', expected, fmt='%01d')
np.savetxt('predicted.txt',predicted , fmt='%01d')
print(cm)
print(expected.shape)
print(predicted.shape)
cm.stats()

print("***************************************************************")



(29704,)
(29704,)
population: 29704
P: 14254
N: 15450
PositiveTest: 15001
NegativeTest: 14703
TP: 13315
TN: 13764
FP: 1686
FN: 939
TPR: 0.934123754736
TNR: 0.890873786408
PPV: 0.887607492834
NPV: 0.936135482555
FPR: 0.109126213592
FDR: 0.112392507166
FNR: 0.0658762452645
ACC: 0.91162806356
F1_score: 0.910271748419
MCC: 0.824370019609
informedness: 0.824997541143
markedness: 0.823742975388
prevalence: 0.479868031242
LRP: 8.56003084855
LRN: 0.073945654558
DOR: 115.761107258
FOR: 0.0638645174454
Predicted  False   True  __all__
Actual                          
False      13764   1686    15450
True         939  13315    14254
__all__    14703  15001    29704
(29704,)
(29704,)
***************************************************************


In [16]:
print(f"Classification report for classifier {model}:\n"
      f"{metrics.classification_report(expected, predicted)}\n")

Classification report for classifier SVC(C=1.0, cache_size=200, class_weight=None, coef0=0.0,
  decision_function_shape='ovr', degree=3, gamma='auto_deprecated',
  kernel='linear', max_iter=-1, probability=False, random_state=None,
  shrinking=True, tol=0.001, verbose=False):
              precision    recall  f1-score   support

           0       0.94      0.89      0.91     15450
           1       0.89      0.93      0.91     14254

   micro avg       0.91      0.91      0.91     29704
   macro avg       0.91      0.91      0.91     29704
weighted avg       0.91      0.91      0.91     29704




In [17]:
from keras.models import Sequential, Model
from keras.layers import Dense, Dropout, Activation, Embedding
from keras.wrappers.scikit_learn import KerasClassifier
import h5py
from keras import callbacks
from keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, CSVLogger
from keras.utils import to_categorical
def build_model():
    # 1. define the network
    model = Sequential()
    #model = Model()
    model.add(Dense(1024,input_dim=41,activation='relu'))  
    model.add(Dropout(0.01))
    model.add(Dense(1))
    model.add(Activation('sigmoid'))
    # try using different optimizers and different optimizer configs
    model.compile(loss='binary_crossentropy',optimizer='adam',metrics=['accuracy'])   
   # model.compile(loss='categorical_crossentropy',optimizer='adam',metrics=['accuracy'])   
    return model

In [18]:
#DNN
checkpointer = callbacks.ModelCheckpoint(filepath="./DNNResult/checkpoint-{epoch:02d}.hdf5", verbose=1, save_best_only=True, monitor='loss')
#csv_logger = CSVLogger('./DNNResult/training_set_dnnanalysis.csv',separator=',', append=False)
model = KerasClassifier(build_fn=build_model, epochs=40, batch_size=64)
#model.fit(traindata, trainlabel, callbacks=[checkpointer,csv_logger])
model.fit(traindata, trainlabel, callbacks=[checkpointer])
#model.save("DNNResult/dnn1layer_model.hdf5")

# make predictions
expected = testlabel
predicted = model.predict(testdata)
# summarize the fit of the model
expected = expected.flatten()
predicted = predicted.flatten()
print(predicted.shape)
print(expected.shape)

cm = ConfusionMatrix(expected, predicted)
expected = np.array(expected)
predicted = np.array(predicted)
cm.print_stats()
np.savetxt('expected.txt', expected, fmt='%01d')
np.savetxt('predicted.txt',predicted , fmt='%01d')
print(cm)
print(expected.shape)
print(predicted.shape)
cm.stats()

print("***************************************************************")


Epoch 1/40
172672/172672 [==============================] - 10s 58us/step - loss: 0.1606 - acc: 0.9356

Epoch 00001: loss improved from inf to 0.16056, saving model to ./DNNResult/checkpoint-01.hdf5
Epoch 2/40
172672/172672 [==============================] - 9s 50us/step - loss: 0.0882 - acc: 0.9685

Epoch 00002: loss improved from 0.16056 to 0.08822, saving model to ./DNNResult/checkpoint-02.hdf5
Epoch 3/40
172672/172672 [==============================] - 9s 50us/step - loss: 0.0703 - acc: 0.9746

Epoch 00003: loss improved from 0.08822 to 0.07028, saving model to ./DNNResult/checkpoint-03.hdf5
Epoch 4/40
172672/172672 [==============================] - 10s 55us/step - loss: 0.0621 - acc: 0.9774

Epoch 00004: loss improved from 0.07028 to 0.06211, saving model to ./DNNResult/checkpoint-04.hdf5
Epoch 5/40
172672/172672 [==============================] - 10s 59us/step - loss: 0.0572 - acc: 0.9790

Epoch 00005: loss improved from 0.06211 to 0.05721, saving model to ./DNNResult/checkpoint

__all__    15349  14355    29704
(29704,)
(29704,)
***************************************************************
